# 원재료 HS 4자리 판정 모델 (최종, 2026-11-10)
화장품 성분표의 원료명(국문 또는 영문 INCI)을 넣으면 **HS 4자리 1~3순위와 판정상태(자동 확정 / 확인 필요)**를 알려 줌. 원산지 사전점검(CTH)에서 원료의 HS를 채우는 데 씀.

**판정 순서**: ① 성분명 정리·식약처 표준명 찾기 → ② 학습 데이터에 같은 이름이 있으면 그 정답 → ③ 이름 모델(국문·영문 문자 TF-IDF + 로지스틱 + HS 품목표 용어 규칙) → ④ 분자 구조 규칙(PubChem 구조로 제29류 호 결정) → ⑤ '베이스·벌크·반제품' 같은 이름은 제품과 같은 호 가능성으로 표시

**데이터**: 정답표 = 구글 시트 '원재료 모델 테스트셋' 시트1 814행(학습 722 / 시험 A 92, R열 데이터구분) · 식약처 화장품 원료성분 DB · PubChem 구조

### 최신 성능
| 평가 | 원료 수 | 정확도 | Macro-F1 | 3순위 안 | 비고 |
|---|---|---|---|---|---|
| 교차검증(4겹, 같은 원료는 한쪽에만) | 605 | 0.896 | 0.782 | 0.954 | 규칙 보완에 참고한 데이터라 다소 낙관적 |
| 시험 A (처음부터 떼어 둔 행) | 92 | 0.837 | 0.611 | 0.891 | 같은 크기로 90번 다시 떼면 정확도 0.78±0.04 |
| 시험 B (실제 성분표, 정답 검증 완료) | 110 | 0.836 | 0.752 | 0.882 | 분류불가 10종 제외 |
| 참고: NCS 처방 원료(정답 팀 확인 전) | 174 | 0.63 | - | - | 판단이 갈리는 혼합물·코팅 원료 빼면 0.76 |
| 원산지 CTH 판정(실제 제품×성분) | 672쌍 | 100% 일치 | - | - | 위험 오판 0건 |

- 잘하는 것: 식물 추출물(1302), 구조가 분명한 제29류 화합물(구조 규칙 적용 원료 정답률 0.60 → 0.97)
- 약한 것: 계면활성제(3402), 혼합 조제품(3824), 4급 암모늄(2923), 무기염. 판정상태가 '확인 필요'인 원료는 사람이 확인
- 화장품 호(3304·3305 등)와 헷갈리는 오답은 거의 없어 원산지 판정에는 영향이 적음

### 사용 순서
1. **준비** 칸 실행 (구글 로그인 창이 뜨면 허용, 1~2분)
2. **최종 모델 사용** 칸에서 원료명이나 성분표를 바꿔 실행
3. 필요할 때만 **평가** 칸 실행 (2~3분)

예전 시제품·실험 칸은 「원재료 HS4 판정 모델 (실험 기록).ipynb」로 옮김

In [1]:
# 준비: 드라이브(원재료 모델/00_공통)에서 코드·데이터를 내려받고 최종 모델을 만듦
import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'])
from google.colab import auth; auth.authenticate_user()
import google.auth; from googleapiclient.discovery import build
_d = build('drive', 'v3', credentials=google.auth.default()[0])
_q = lambda n, p: _d.files().list(q=f"name='{n}' and '{p}' in parents and trashed=false", fields='files(id)').execute()['files'][0]['id']
exec(_d.files().get_media(fileId=_q('00_setup.py', _q('00_공통', '1bNwzZnTPU0T64xoycbgGE2iJAlTpeTgo'))).execute().decode('utf-8'))   # 시트1·식약처·PubChem 등 준비
for fn in ['origin_check.py', 'hs4_final.py']: open(fn, 'wb').write(get(child(fn, common)))
import warnings, pandas as pd; warnings.filterwarnings('ignore'); pd.set_option('display.width', 200)
from hs4_final import HS4Final
model = HS4Final('/content/rawmodel')
print('최종 모델 준비 완료')

준비 완료 | 시트1 814 행 (학습 722 / 시험A 92 ) | 실제 성분 1078 개
최종 모델 준비 완료


## 최종 모델 사용
- 원료명 하나: 국문명이나 영문 INCI명 하나를 넣음
- 성분표 하나: 전성분을 쉼표로 이어서 붙여 넣음 ('1, 2-헥산다이올'처럼 숫자 사이 쉼표는 이름으로 처리)
- 판정상태가 '확인 필요'면 1~3순위를 보고 사람이 확인. '반제품·베이스 의심'은 제품과 같은 호일 수 있어 원산지 판정에 중요

In [2]:
원료명 = '나이아신아마이드'      # ← 바꿔서 실행 (예: 'Sodium Hyaluronate', '병풀추출물')
pd.DataFrame([model.one(원료명)])

,입력,찾은 이름(국문),찾은 이름(영문 INCI),이름 찾기,HS4 1순위,HS4 2순위,HS4 3순위,판정상태
0,나이아신아마이드,나이아신아마이드,Niacinamide,식약처 일치,2924,2936,2933,확인 필요(점수 낮음)


In [3]:
성분표 = '정제수, 글리세린, 부틸렌글라이콜, 나이아신아마이드, 1, 2-헥산다이올, 병풀추출물, 다이메티콘, 카보머, 트로메타민, 향료'   # ← 바꿔서 실행
결과 = model.bom(성분표)
print('자동 확정', (결과['판정상태'].str.startswith('자동')).sum(), '/', len(결과), '| 확인 필요', (결과['판정상태'].str.startswith('확인')).sum())
결과

자동 확정 9 / 10 | 확인 필요 1


,입력,찾은 이름(국문),찾은 이름(영문 INCI),이름 찾기,HS4 1순위,HS4 2순위,HS4 3순위,판정상태
0,정제수,정제수,Water,식약처 일치,2853,3301,1302,자동 확정(학습 데이터 일치)
1,글리세린,글리세린,Glycerin,식약처 일치,2905,2922,2932,자동 확정(분자 구조 규칙)
2,부틸렌글라이콜,부틸렌글라이콜,Butylene Glycol,식약처 일치,2905,2909,2906,자동 확정(학습 데이터 일치)
3,나이아신아마이드,나이아신아마이드,Niacinamide,식약처 일치,2924,2936,2933,확인 필요(점수 낮음)
4,"1,2-헥산다이올","1,2-헥산다이올","1,2-Hexanediol",식약처 일치,2905,2906,2909,자동 확정(분자 구조 규칙)
5,병풀추출물,병풀추출물,Centella Asiatica Extract,식약처 일치,1302,1211,3301,자동 확정
6,다이메티콘,다이메티콘,Dimethicone,식약처 일치,3910,2914,2931,자동 확정(학습 데이터 일치)
7,카보머,카보머,Carbomer,식약처 일치,3906,1302,2836,자동 확정(학습 데이터 일치)
8,트로메타민,트로메타민,Tromethamine,식약처 일치,2922,2921,2929,자동 확정(분자 구조 규칙)
9,향료,향료,Flavor,식약처 일치,3301,1302,2932,자동 확정(학습 데이터 일치)


## 평가: 최종 시험(06) 다시 실행
- 시험 A: 시트1에서 처음부터 떼어 둔 92행 (모델·규칙 수정에 쓰지 않음)
- 시험 B: 실제 성분표 120종, 정답 2026-11-10 검증 완료(00_공통/test_b_v2.csv '확정 HS4'), 분류불가 10종 제외 110종 채점
- 결과는 /content/rawmodel/06_final_test/ 의 final_test_results.csv, test_a_predictions.csv, test_b_predictions.csv

In [4]:
import os; os.chdir('/content/rawmodel'); open('06_final_test/run.py', 'wb').write(get(child('run_06.py', common)))
os.chdir('/content/rawmodel/06_final_test'); exec(open('run.py', encoding='utf-8').read()); os.chdir('/content/rawmodel')

                                           시험                      방법  accuracy  macro_f1  weighted_f1  top3_accuracy  lenient_accuracy
0                                 시험A(시트 92행)                   이름 모델     0.739     0.435        0.709          0.804             0.739
1                                 시험A(시트 92행)           이름 모델 + 구조 규칙     0.837     0.611        0.824          0.891             0.837
2  시험B(실제 성분 110종, 정답 검증 완료, 초안 때 규칙 수정에 참고함)         정리 전 이름 → 이름 모델     0.736     0.620        0.707            NaN               NaN
3  시험B(실제 성분 110종, 정답 검증 완료, 초안 때 규칙 수정에 참고함)  성분명 정리 → 이름 모델 + 구조 규칙     0.836     0.752        0.809          0.882               NaN
시험B 식약처 찾기: 정리 전 0.845 → 정리 후 0.855 | 정답 확정 110
시험B 틀린 것 18 개 (구조 적용 25 개)
                    입력 국문명    정답 정리 후 1순위 정리 후 찾기   구조적용
2                       향료  3302     3301  식약처 일치  False
4                 나이아신아마이드  2936     2924  식약처 일치  False
7                   인산완충용액  3824     2932    못 찾음  False
24         PEG-40